# Competitor Intelligence Report Generator

Give this tool any company URL and it will:
1. Scrape the company's website to understand what they do
2. Use **Tavily** to find their top competitors in real time
3. Scrape each competitor's website
4. Use an LLM to extract structured profiles from all companies
5. Generate a full **Competitive Intelligence Report** including:
   - Company & competitor profiles
   - Side-by-side competitive matrix
   - Market standing analysis
   - Strategic recommendations

---

**You can use any LLM you want.** Just change the model name in Cell 3.  
The wrapper automatically routes to the correct provider.


## Cell 1 — Check API Keys
Make sure your `.env` file is loaded and the keys you need are present.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from dotenv import load_dotenv
load_dotenv(override=True)

keys_to_check = {
    "TAVILY_API_KEY":  "Tavily  (required for competitor search)",
    "GROQ_API_KEY":    "Groq    (free — recommended default)",
    "GOOGLE_API_KEY":  "Gemini  (free)",
    "OPENAI_API_KEY":  "OpenAI  (paid)",
    "MISTRAL_API_KEY": "Mistral (free tier)",
}

print("API Key Status:\n")
for env_var, label in keys_to_check.items():
    val = os.getenv(env_var, "")
    status = "OK" if val and "your_" not in val else "NOT SET"
    print(f"  {status:8s}  {label}")

## Cell 2 — Available Models
See every model you can use and which provider it routes to.

In [ ]:
from config import MODEL_TO_PROVIDER, FAST_MODEL, SMART_MODEL

print("Available models:\n")
for model, provider in MODEL_TO_PROVIDER.items():
    tag = ""
    if model == FAST_MODEL:  tag = "  <-- default FAST model (LLM 1 & 2)"
    if model == SMART_MODEL: tag = "  <-- default SMART model (LLM 3 — report)"
    print(f"  {provider:10s}  {model}{tag}")

## Cell 3 — Configure Your Run

Set the company URL, name, and the models you want to use.

- **`FAST_MODEL`** handles extraction (LLM 1 & 2) — use a smaller, cheaper model here  
- **`SMART_MODEL`** generates the final report (LLM 3) — use a stronger model here  
- Change either model to any name from Cell 2 above

In [ ]:
# ----------------------------------------------------------------
# CONFIGURE HERE
# ----------------------------------------------------------------

COMPANY_URL  = "https://www.stripe.com"      # <-- change to any company URL
COMPANY_NAME = "Stripe"                       # <-- company name (used for search)

FAST_MODEL_OVERRIDE  = "llama-3.1-8b-instant"    # used for extraction (LLM 1 & 2)
SMART_MODEL_OVERRIDE = "llama-3.3-70b-versatile" # used for report generation (LLM 3)

MAX_COMPETITORS = 4   # how many competitors to find and analyse (3-5 recommended)

# ----------------------------------------------------------------
print(f"Target company : {COMPANY_NAME}")
print(f"URL            : {COMPANY_URL}")
print(f"Fast model     : {FAST_MODEL_OVERRIDE}")
print(f"Smart model    : {SMART_MODEL_OVERRIDE}")
print(f"Max competitors: {MAX_COMPETITORS}")

## Cell 4 — Run the Pipeline

This cell runs all 6 steps and prints progress as it goes.  
Typical run time: **30–90 seconds** depending on how many competitors are found.

In [ ]:
from report import run_competitor_intelligence

intelligence_report = run_competitor_intelligence(
    company_url=COMPANY_URL,
    company_name=COMPANY_NAME,
    fast_model=FAST_MODEL_OVERRIDE,
    smart_model=SMART_MODEL_OVERRIDE,
    max_competitors=MAX_COMPETITORS,
)

## Cell 5 — Display the Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(intelligence_report))

## Cell 6 — Save the Report to a File (Optional)

In [ ]:
import re
from datetime import datetime

safe_name = re.sub(r'[^\w]', '_', COMPANY_NAME.lower())
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
filename  = f"{safe_name}_competitor_report_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(intelligence_report)

print(f"Report saved to: {filename}")

---

## Quick Reference — Model Options

| Provider | Model | Cost | Good for |
|----------|-------|------|----------|
| Groq | `llama-3.1-8b-instant` | Free | Fast extraction (LLM 1 & 2) |
| Groq | `llama-3.3-70b-versatile` | Free | Full report (LLM 3) |
| Groq | `gemma2-9b-it` | Free | Fast extraction |
| Gemini | `gemini-2.0-flash` | Free | Fast + capable |
| Gemini | `gemini-1.5-pro` | Free | Strong report generation |
| OpenAI | `gpt-4o-mini` | Paid | Fast extraction |
| OpenAI | `gpt-4o` | Paid | Best report quality |
| Ollama | `phi3:mini` | Local | Fully offline, no cost |
| Ollama | `llama3.2` | Local | Fully offline, no cost |
